# Data Engineering for the Machine Learning Process

**Author**: Nikolas Antoniou

**Supervisor**: Dr. Demetris Trihinas

**Project**: Butterfly Classification (Cyprus)

This notebook is the data engineering stage of the pipeline we need. It turns the audited modelling table into specific datasdets to be used for later training.

_______________

## 1. Importing Libraries & Data

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
import re
import requests
from pathlib import Path
import sys

In [2]:
# defining roots and sources
ROOT = Path.cwd().parents[0] if Path.cwd().name == "Analysis" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [3]:
# importing insects
import insects
from insects import config

In [4]:
print(f"insects v{insects.__version__} loaded from {Path(insects.__file__).parent}")

insects v0.1.0 loaded from /Users/nicolasdilegardia/Desktop/insects/src/insects


In [5]:
print(f"repo root : {config.ROOT}")

repo root : /Users/nicolasdilegardia/Desktop/insects


In [6]:
from insects.inat import INatClient
from insects.taxonomy import parse_ancestors
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)

### 1.2 Load Modelling Table

The dataset **insects_clean.csv** is one of the derivables of the audit stage. Each row is an image which survived the rules S4-S11. They have to be treated as a read-only input. All outputs will be written to new files to avoid overwritings. The notebook can run again at any moment.

In [7]:
insects_df = pd.read_csv(config.CLEAN_CSV)

In [8]:
assert config.LABEL_COLUMN in insects_df.columns, "not the cleaned table"
assert insects_df[config.GROUP_KEY].notna().all(), "missing observation ids"

In [9]:
print(f"{len(insects_df):,} image rows\n")
print(f"{insects_df[config.LABEL_COLUMN].nunique():,} species\n")
print(f"{insects_df[config.GROUP_KEY].nunique():,} observations\n")
insects_df.head()

16,561 image rows

266 species

9,401 observations



,observation_id,label_species,taxon_name,photo_url,photo_url_medium,license_code,attribution,photographer,location,district
0,387625264,Utetheisa pulchella,Utetheisa pulchella,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,cc-by-nc,"(c) Reflectitur Photons, some rights reserved ...",Reflectitur Photons,"Nicosia, Cyprus",nicosia
1,387625095,Macroscytus brunneus,Macroscytus brunneus,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,cc-by-nc,"(c) Reflectitur Photons, some rights reserved ...",Reflectitur Photons,"Nicosia, Cyprus",nicosia
2,387625063,Vespa orientalis,Vespa orientalis,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,cc-by-nc,"(c) Reflectitur Photons, some rights reserved ...",Reflectitur Photons,"Nicosia, Cyprus",nicosia
3,387618636,Phoracantha recurva,Phoracantha recurva,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,cc-by-nc,"(c) Reflectitur Photons, some rights reserved ...",Reflectitur Photons,"Nicosia, Cyprus",nicosia
4,387618636,Phoracantha recurva,Phoracantha recurva,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,cc-by-nc,"(c) Reflectitur Photons, some rights reserved ...",Reflectitur Photons,"Nicosia, Cyprus",nicosia


______________

## 2. Deriving the Butterfly Superfamily

### 2.1 What is defined as a "butterfly"?

This project is concentrated around the butterfly species which live in Cyprus. Thus, the first data engineering question is the **operational definition** of the class. This is a pure taxonomic question, not a string-match. Getting it wrong would silently destroy the future results.

**Butterfly** is not a rank but a Superfamily. Specifically the **Papilionoidea** superfamily. There is no property of a Latin binomial that identifies one. `Pieris rapae` is a butterfly and `Utetheisa pulchella` is a moth, but as *strings* they are indistinguishable:

- both are `Genus species`, 
- both are Lepidoptera, 
- neither name encodes its lineage. 

Any rule built on name patterns, suffixes, or keyword lists would be
folklore dressed as a filter.

What *does* define butterflies is descent from a single common ancestor:
the superfamily **Papilionoidea**.

Historically, butterflies were split across three different superfamilies:<br>
- Papilionoide (true butterflies),
- Hesperioidea (skippers),
- Hedyloidea (moths).<br>
**Heikkilä et al. (2012)** showed with combined morphological and molecular data that this arrangement makes butterflies *paraphyletic*: <br>
Papilionidae is sister to the skippers, hedylids and all remaining butterflies. <br>
Their proposal — now standard — is to place all of these families in a **single superfamily, Papilionoidea**.

**Kawahara et al. (2023)** is the current authority on the group: a phylogenomic
tree from 391 genes across ~2,300 species covering 92% of butterfly genera. It
confirms this circumscription, dates the origin of butterflies at ~100 Ma, and
refines relationships within it (at least 36 tribes require reclassification).

> **Sources**
> - Heikkilä, M., Kaila, L., Mutanen, M., Peña, C. & Wahlberg, N. (2012).
>   *Cretaceous origin and repeated tertiary diversification of the redefined
>   butterflies.* **Proceedings of the Royal Society B**, 279(1731), 1093–1099.
>   [doi:10.1098/rspb.2011.1430](https://doi.org/10.1098/rspb.2011.1430)
> - Kawahara, A. Y., Storer, C., Carvalho, A. P. S., *et al.* (2023).
>   *A global phylogeny of butterflies reveals their evolutionary history,
>   ancestral hosts and biogeographic origins.* **Nature Ecology & Evolution**,
>   7, 903–913.
>   [doi:10.1038/s41559-023-02041-9](https://doi.org/10.1038/s41559-023-02041-9)
> - iNaturalist taxonomy, taxon **47224** (Papilionoidea):
>   [inaturalist.org/taxa/47224](https://www.inaturalist.org/taxa/47224)

iNaturalist implements exactly this concept. Querying the API for taxon
`47224` (verified 2026-08-24) returns rank `superfamily` with seven child
families:<br>

| Family | Common name |
|---|---|
| Papilionidae | Swallowtails and Parnassians |
| Pieridae | Whites, Yellows and Sulphurs |
| Nymphalidae | Brush-footed Butterflies |
| Lycaenidae | Gossamer-winged Butterflies |
| Riodinidae | Metalmark Butterflies |
| Hesperiidae | Skippers |
| Hedylidae | American Moth-Butterflies |

**Caveat, stated up front:** this is butterflies *sensu lato*. It includes
skippers and hedylids, which colloquial usage often excludes. That is a
deliberate choice — we follow the phylogenetic definition rather than the
vernacular one, and it is already encoded in `config.BUTTERFLY_TAXON_ID`. For
Cyprus the practical effect is limited to Hesperiidae; Hedylidae is Neotropical
and cannot occur in our data.

### 2.2 The processing rule

Every iNaturalist observation carries `taxon.ancestor_ids` — the full ordered
chain from kingdom down to the identified taxon. Clade membership is therefore
a **set-membership test**, exact and cheap:

    is_butterfly  ⇔  47224 ∈ ancestor_ids

Verified end-to-end on a known case:

    Anthocharis cardamines (Orange Tip)  →  taxon 51495, rank species
    ancestors: [48460, 1, 47120, 372739, 47158, 184884, 47157, 47224, ...]
                                                              ↑ 47224 present
    is_butterfly = True ✓

This is precisely what `src/insects/taxonomy.py` already implements, so we reuse
it rather than reinventing it. The complication is that the ancestry chain was
dropped from `insects_clean.csv` by an earlier version of the pipeline. Section
2.3 recovers it, for the 266 distinct species only — 266 lookups instead of
16,561, because the flag is a property of the *species*, not of the photo.

The pipeline for this section is therefore:

    insects_clean.csv
      → recover photo_id from the photo URL            (2.3, offline)
      → resolve 266 species names → taxon_id           (2.4, API)
      → join ancestry → is_butterfly + family/genus    (2.5)
      → filter → Data/ButterfliesOnly/butterflies.csv  (2.6)

### 2.3 Recovering *photo_id*<br>

`insects_clean.csv` is missing `photo_id`, even though `clean.py`'s `OUT_COLS`
promises it: the file on disk was written by an earlier version of the pipeline.
We need it, because `photo_id` is the only **stable, globally unique key for a
single image file** — and therefore the only sane basis for naming the files we
download in Section 3.

`observation_id` cannot do that job: one observation may carry several photos,
so it is not unique per image. Filenames derived from it would collide.

We do **not** need the API for this. iNaturalist photo URLs are structurally
templated, and the id is already inside the string we hold:

    https://inaturalist-open-data.s3.amazonaws.com/photos/709822987/medium.jpg
                                                   └────┬────┘
                                                     photo_id

So the recovery is a regex over a column, not 16,561 network calls. **Prefer
parsing data you already have over re-fetching it** — it is faster, it is
reproducible offline, and it cannot drift from the source table.

The two assertions encode what must be true if the extraction worked: every row
yields an id, and no id repeats. A duplicate would mean the same image file
appears twice, which is the leakage rule S5 was written to prevent — so if this
fires, it is a real finding, not a nuisance.

In [10]:
PHOTO_ID_RE = re.compile(r"/photos/(\d+)/")

In [11]:
def photo_id_from_url(url) -> int | None:
    """Extract iNaturalist's image id from url"""
    match = PHOTO_ID_RE.search(url) if isinstance(url, str) else None
    return int(match.group(1)) if match else None

In [12]:
insects_df["photo_id"] = insects_df["photo_url"].map(photo_id_from_url)
n_missing = insects_df["photo_id"].isna().sum()
n_unique = insects_df["photo_id"].nunique()

In [13]:
print(f"Extracted: {len(insects_df) - n_missing:,} out of {len(insects_df):,} records")
print(f"Unmatched: {n_missing:,}")
print(f"Unique: {n_unique:,}")

Extracted: 16,561 out of 16,561 records
Unmatched: 0
Unique: 16,561


In [14]:
assert n_missing == 0, "some photo URLs did not match the expected pattern"
assert n_unique == len(insects_df), "duplicate photo files survived cleaning (S5)"

In [15]:
insects_df[["observation_id", "photo_id", "label_species"]].head()

,observation_id,photo_id,label_species
0,387625264,709822987,Utetheisa pulchella
1,387625095,709822622,Macroscytus brunneus
2,387625063,709822553,Vespa orientalis
3,387618636,709810271,Phoracantha recurva
4,387618636,709810366,Phoracantha recurva


In [16]:
insects_df[["observation_id", "photo_id", "label_species"]].tail()

,observation_id,photo_id,label_species
16556,2089558,2500268,Crocothemis erythraea
16557,2089558,2500269,Crocothemis erythraea
16558,1927897,2358111,Vespa orientalis
16559,1927897,2358112,Vespa orientalis
16560,1530123,1895794,Sphodromantis viridis


### 2.4 Species to Taxa

We will need the *ancestor_ids* for each of the species and because the clean table no longer suports it we will ask the iNaturalist taxonomy for it.

Now we resolve the species, not the images. Results are written to `Data/raw/species_taxonomy.csv`, and re-running fetches only names not already present. A kernel restart therefore costs nothing, and the cell is safe to re-run at any point — the same top-up-don't-trust pattern `taxonomy.py` uses for its own taxon cache. (`Data/raw/` is gitignored: it is regenerable from code, so it is derived data, not source.)

In [17]:
from tqdm.auto import tqdm

In [18]:
SPECIES_TAXONOMY_CSV = config.RAW / "species_taxonomy.csv"
SPECIES_COLS = ["label_species",
                "taxon_id",
                "taxon_rank",
                "ancestor_ids",
                "observations_count"]

In [19]:
# function for resolving
def resolve_species(name: str, client:  INatClient) -> dict | None:
    """Look for one species name,
        Return taxon record,
        None if no match
    """
    
    payload = client.get("taxa", {"q": name, "rank": "species", "per_page": 10})
    
    for result in payload.get("results", []):
        if result.get("name") == name:              # exact match only
            ancestors = result.get("ancestor_ids") or []
            return {
                "label_species": name,
                "taxon_id": result["id"],
                "taxon_rank": result.get("rank"),
                "ancestor_ids": "|".join(str(a) for a in ancestors),
                "observations_count": result.get("observations_count"),
            }
    return None

In [20]:
# building species table
def build_species_table(names, cache_path=SPECIES_TAXONOMY_CSV) -> pd.DataFrame:
    """Resolve every name, fetching only those absent from the cache."""
    if cache_path.exists():
        cached = pd.read_csv(cache_path, dtype={"ancestor_ids": str})
    else:
        cached = pd.DataFrame(columns=SPECIES_COLS)

    known = set(cached["label_species"])
    missing = sorted(set(names) - known)
    print(f"{len(known):,} cached, {len(missing):,} to fetch")

    if not missing:
        return cached

    client = INatClient()
    fetched, unresolved = [], []

    for name in tqdm(missing, desc="resolving species"):
        record = resolve_species(name, client)
        (fetched if record else unresolved).append(record or name)

    out = pd.concat([cached, pd.DataFrame(fetched)], ignore_index=True)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(cache_path, index=False)

    print(f"resolved {len(fetched):,} / {len(missing):,}")
    if unresolved:
        print(f"UNRESOLVED ({len(unresolved)}): {unresolved}")

    return out

In [21]:
species_names = insects_df[config.LABEL_COLUMN].dropna().unique()
species_tax = build_species_table(species_names)

species_tax.head()

266 cached, 0 to fetch


,label_species,taxon_id,taxon_rank,ancestor_ids,observations_count
0,Achyra nudalis,371398,species,48460|1|47120|372739|47158|184884|47157|49682|...,2879
1,Acmaeodera flavolineata,891926,species,48460|1|47120|372739|47158|184884|47208|71130|...,76
2,Acmaeodera saxicola,773015,species,48460|1|47120|372739|47158|184884|47208|71130|...,53
3,Acontia lucida,415493,species,48460|1|47120|372739|47158|184884|47157|47607|...,7093
4,Acrida ungarica,211055,species,48460|1|47120|372739|47158|184884|47651|67688|...,7216


### 2.5 Application of the Clade Test

The main rule:<br>
is_butterfly  ⇔  47224 (Papilionoidea) ∈ ancestor_ids<br>
is_moth       ⇔  47157 (Lepidoptera) ∈ ancestor_ids  AND  47224 ∉ ancestor_ids<br>

Moths became our negative class. The seperation of a butterfly from a moth is one of the most crucial steps.

In [22]:
from insects.taxonomy import build_taxon_table, enrich

In [23]:
cached_taxa = None
if config.TAXONOMY_CSV.exists():
    cached_taxa = pd.read_csv(config.TAXONOMY_CSV).set_index("taxon_id")

In [24]:
taxa = build_taxon_table(species_tax, cached_taxa)
taxa.to_csv(config.TAXONOMY_CSV)

INFO taxon cache covers all 1,004 required taxa


In [25]:
species_enriched = enrich(species_tax, taxa)

n_bf = int(species_enriched["is_butterfly"].sum())
n_moth = int(species_enriched["is_moth"].sum())

print(f"butterfly species : {n_bf:,} / {len(species_enriched):,}")
print(f"moth species      : {n_moth:,}")
print(f"other insects     : {len(species_enriched) - n_bf - n_moth:,}")

species_enriched.loc[
    species_enriched["is_butterfly"] == 1,
    ["label_species", "family", "genus"]
].sort_values("family").head(10)

butterfly species : 33 / 266
moth species      : 71
other insects     : 162


,label_species,family,genus
46,Carcharodus alceae,Hesperiidae,Carcharodus
246,Thymelicus acteon,Hesperiidae,Thymelicus
190,Pelopidas thrax,Hesperiidae,Pelopidas
103,Gegenes pumilio,Hesperiidae,Gegenes
143,Lycaena phlaeas,Lycaenidae,Lycaena
241,Tarucus balkanica,Lycaenidae,Tarucus
204,Polyommatus icarus,Lycaenidae,Polyommatus
144,Lycaena thersamon,Lycaenidae,Lycaena
142,Luthrodes galba,Lycaenidae,Luthrodes
136,Leptotes pirithous,Lycaenidae,Leptotes


In [26]:
JOIN_COLS = (["label_species", "taxon_id"]
             + config.RANK_COLUMNS
             + ["is_butterfly", "is_moth"])

insects_df = insects_df.merge(
    species_enriched[JOIN_COLS],
    on="label_species",
    how="left",
    validate="many_to_one",
)

assert insects_df["is_butterfly"].notna().all(), "some photos got no clade flag"

print(f"butterfly photos : {int(insects_df['is_butterfly'].sum()):,} / {len(insects_df):,}")
print(f"moth photos      : {int(insects_df['is_moth'].sum()):,}")

butterfly photos : 2,900 / 16,561
moth photos      : 4,070


### 2.6 Butterfly Subset

The filter is now a single comparison against a derived, cited flag rather than
a pattern match on names — the whole point of Sections 2.1–2.5.

Two columns are dropped on the way out. `is_butterfly` is `1` for every row here
and `is_moth` is `0` for every row: constant columns carry no information and
invite mistakes downstream (a stray `== 0` filter that silently returns nothing).
The clade rule is documented above; it does not need to be re-stated 2,900 times.

One column is added. `source` records **where each row came from** — `cyprus_inat`
for these rows, and `global_inat` for the images harvested in Section 3. It costs
one string per row and buys the ability to train on the full set while evaluating
on Cyprus data alone. Provenance is far easier to record now than to reconstruct
later, and reconstructing it after a merge is usually impossible.

The output directory is created from code, not by hand. Anyone who clones the
repo and runs this notebook gets the same tree, which is the difference between
a reproducible pipeline and one that works on one laptop.

In [27]:
BUTTERFLIES_DIR = config.DATA / "ButterfliesOnly"
BUTTERFLIES_CSV = BUTTERFLIES_DIR / "butterflies.csv"

BUTTERFLY_COLS = [
    "observation_id", "photo_id", "label_species", "taxon_name", "taxon_id",
    "order", "superfamily", "family", "subfamily", "tribe", "genus",
    "photo_url", "photo_url_medium",
    "license_code", "attribution", "photographer",
    "location", "district", "source",
]

butterflies = insects_df[insects_df["is_butterfly"] == 1].copy()
butterflies["source"] = "cyprus_inat"
butterflies = butterflies[BUTTERFLY_COLS].reset_index(drop=True)

BUTTERFLIES_DIR.mkdir(parents=True, exist_ok=True)
butterflies.to_csv(BUTTERFLIES_CSV, index=False)

print(f"wrote {len(butterflies):,} rows -> {BUTTERFLIES_CSV.relative_to(config.ROOT)}")
print(f"{butterflies['label_species'].nunique()} species, "
      f"{butterflies['family'].nunique()} families, "
      f"{butterflies['observation_id'].nunique():,} observations")

butterflies["label_species"].value_counts().head(10)

wrote 2,900 rows -> Data/ButterfliesOnly/butterflies.csv
33 species, 5 families, 1,835 observations


label_species
Vanessa cardui           362
Papilio machaon          290
Glaucopsyche paphos      189
Pieris brassicae         169
Colias crocea            141
Hipparchia cypriensis    122
Pontia daplidice         115
Pieris rapae             107
Lycaena phlaeas          101
Maniola cypricola         97
Name: count, dtype: int64

_______________

## 3. Image Dataset

### 3.1 Harvesting for the images

2,900 photos across 33 classes — a median of 66 per species, and 21 for the
smallest — is below what a fine-grained image classifier needs. We need more
images per class, and they must carry **trustworthy labels**.

In [28]:
from insects.retrieve import flatten

HARVEST_CAP = 500
HARVEST_RAW_CSV = config.RAW / "harvest_raw.csv"

# iter_observations logs one line per page; across 33 species that is noise.
logging.getLogger("insects.inat").setLevel(logging.WARNING)


def harvest_species(taxon_id: int, need: int, client: INatClient,
                    exclude: set) -> list[dict]:
    """Fetch up to `need` new photo rows for one taxon, worldwide."""
    if need <= 0:
        return []

    rows: list[dict] = []
    for obs in client.iter_observations(taxon_id=int(taxon_id), place_id=None):
        for row in flatten(obs):
            if row["photo_id"] not in exclude:
                rows.append(row)
        if len(rows) >= need:
            break

    return rows[:need]

In [29]:
held = butterflies["label_species"].value_counts()
seen_photo_ids = set(butterflies["photo_id"])

targets = species_enriched.loc[
    species_enriched["is_butterfly"] == 1, ["label_species", "taxon_id"]
].sort_values("label_species")

if HARVEST_RAW_CSV.exists():
    harvest_df = pd.read_csv(HARVEST_RAW_CSV, dtype={"ancestor_ids": str})
    print(f"loaded cached harvest: {len(harvest_df):,} rows")
else:
    client = INatClient()
    collected = []

    for t in tqdm(targets.itertuples(index=False), total=len(targets),
                  desc="harvesting"):
        need = HARVEST_CAP - int(held.get(t.label_species, 0))
        rows = harvest_species(t.taxon_id, need, client, seen_photo_ids)
        collected.extend(rows)
        seen_photo_ids.update(r["photo_id"] for r in rows)

    harvest_df = pd.DataFrame(collected)
    harvest_df.to_csv(HARVEST_RAW_CSV, index=False)
    print(f"harvested {len(harvest_df):,} new photo rows")

harvest_df["taxon_name"].value_counts().head(5)

loaded cached harvest: 12,188 rows


taxon_name
Euchloe ausonia           479
Gonepteryx cleopatra      470
Freyeria trochylus        468
Lasiommata maera          466
Anthocharis cardamines    463
Name: count, dtype: int64

In [30]:
harvest_df["taxon_name"].value_counts().tail(5)

taxon_name
Colias crocea helice                 2
Pararge aegeria tircis               1
Papilio machaon hudsonianus          1
Anthocharis cardamines cardamines    1
Lycaena phlaeas daimio               1
Name: count, dtype: int64

### 3.2 Normalising

In [31]:
from insects.clean import add_derived


def derive_subspecies(taxon_name) -> str | None:
    """Third word of a trinomial; None for binomials and higher ranks."""
    if not isinstance(taxon_name, str):
        return None
    words = taxon_name.split()
    return words[2] if len(words) == 3 else None


harvest_norm = add_derived(harvest_df)
harvest_norm["subspecies"] = harvest_norm["taxon_name"].map(derive_subspecies)
harvest_norm["source"] = "global_inat"

butterflies["subspecies"] = butterflies["taxon_name"].map(derive_subspecies)

print(f"harvest rows with a subspecies : {harvest_norm['subspecies'].notna().sum():,}")
print(f"cyprus rows with a subspecies  : {butterflies['subspecies'].notna().sum():,}")

assert harvest_norm["label_species"].isin(targets["label_species"]).all(), \
    "harvest returned a species outside the 33 target classes"

harvest rows with a subspecies : 124


cyprus rows with a subspecies  : 1


In [32]:
RANK_JOIN = ["label_species"] + config.RANK_COLUMNS

harvest_norm = harvest_norm.merge(
    species_enriched[RANK_JOIN],
    on="label_species",
    how="left",
    validate="many_to_one",
)

COMBINED_COLS = [
    "observation_id", "photo_id",
    "label_species", "subspecies", "taxon_name", "taxon_id", "taxon_rank",
    *config.RANK_COLUMNS,
    "photo_url", "photo_url_medium",
    "license_code", "attribution", "photographer",
    "location", "district", "source",
]

butterflies_all = pd.concat(
    [butterflies.reindex(columns=COMBINED_COLS),
     harvest_norm.reindex(columns=COMBINED_COLS)],
    ignore_index=True,
)

assert butterflies_all["photo_id"].is_unique, "duplicate images in the combined set"

print(f"combined: {len(butterflies_all):,} photos, "
      f"{butterflies_all['label_species'].nunique()} species, "
      f"{butterflies_all['observation_id'].nunique():,} observations")
print(butterflies_all["source"].value_counts().to_string())


combined: 15,088 photos, 33 species, 9,434 observations
source
global_inat    12188
cyprus_inat     2900


### 3.3 Materialising

The dataset so far indexes images by **URL**. That is not something a model can be trained on: every epoch would re-fetch 15,088 files over the network, the run would be unreproducible (URLs rot, licences change, records get deleted), and it would be abusive of the host. This is the objection raised in supervision, and it is correct.

The resolution is **not** to put images inside the CSV. It is to separate the two concerns:

- **the bytes** live in object storage, as ordinary `.jpg` files;
- **the table** becomes an *index* over them — one row per image, carrying the local path, the label, the licence and the attribution.

That index is called a **manifest**, and it is the interface between the data engineering stage and the training stage.

Files are laid out as `images/<label_species>/<photo_id>.jpg` — the **`ImageFolder` convention** used by `torchvision` and Keras, where the directory name *is* the class label, so a loader can be pointed at the root and infer the classes with no extra configuration. `photo_id` names the file because it is globally unique (verified in 2.3), so no two images can collide.

The downloader lives in `src/insects/download.py`, **not in this notebook**: it is reusable, runnable from the command line, and the notebook's job is the narrative rather than the machinery. It is resumable and idempotent, because fetching ~15,000 files takes tens of minutes and *will* be interrupted. Three mechanisms make that safe:

1. **Skip work already recorded** — the manifest is the ledger of what succeeded.
2. **Write atomically** — bytes go to a temporary `.part` file, renamed into place only once the transfer completes, so a file at its final path is always whole. Without this, an interrupted transfer leaves a truncated `.jpg` that "exists" and is therefore never re-fetched: a corrupt image silently poisoning the training set.
3. **Checksum every file** — a SHA-256 recorded at download time supports integrity checks and reveals byte-identical duplicates that arrived under different photo ids.

First, the table that drives all of this is persisted. `butterflies_all` has so far existed only in memory; the downloader consumes a file, and a dataset that vanishes on kernel restart is not a deliverable.

In [33]:
BUTTERFLIES_ALL_CSV = BUTTERFLIES_DIR / "butterflies_all.csv"

butterflies_all.to_csv(BUTTERFLIES_ALL_CSV, index=False)
print(f"wrote {len(butterflies_all):,} rows -> "
      f"{BUTTERFLIES_ALL_CSV.relative_to(config.ROOT)}")

balance = butterflies_all["label_species"].value_counts()
print(f"\nclass balance  max {balance.max()} | median {int(balance.median())} | "
      f"min {balance.min()}  ->  {balance.max() / balance.min():.1f}:1")
print("(before harvesting it was 362 | 66 | 21  ->  17.2:1)")

balance.tail(5)

wrote 15,088 rows -> Data/ButterfliesOnly/butterflies_all.csv

class balance  max 500 | median 500 | min 116  ->  4.3:1
(before harvesting it was 362 | 66 | 21  ->  17.2:1)


label_species
Gegenes pumilio          475
Glaucopsyche paphos      212
Luthrodes galba          153
Hipparchia cypriensis    132
Maniola cypricola        116
Name: count, dtype: int64

#### Estimating the download before running it

15,088 images at an unknown average size is an unknown quantity of disk, time and bandwidth. We measure a sample before committing, because "how large is this actually?" is a question to answer *before* a long job, not during it.

An HTTP `HEAD` request returns a response's headers **without its body**, so `Content-Length` gives each file's size for a few hundred bytes of traffic. Thirty samples are enough to size a decision that only needs to be right to within a factor of two.

In [34]:
sample = butterflies_all["photo_url_medium"].sample(30, random_state=42)

probe = requests.Session()
probe.headers.update({"User-Agent": config.USER_AGENT})

sizes = []
for url in sample:
    resp = probe.head(url, timeout=15, allow_redirects=True)
    length = resp.headers.get("Content-Length")
    if length is not None:
        sizes.append(int(length))

mean_bytes = np.mean(sizes)
total_gb = mean_bytes * len(butterflies_all) / 1024**3

print(f"sampled      : {len(sizes)}/30 images")
print(f"mean size    : {mean_bytes/1024:.0f} KB")
print(f"estimated    : {total_gb:.2f} GB for {len(butterflies_all):,} images")
print(f"at 8 workers : roughly {len(butterflies_all)/8/6/60:.0f}-{len(butterflies_all)/8/2/60:.0f} minutes")

sampled      : 30/30 images
mean size    : 132 KB
estimated    : 1.90 GB for 15,088 images
at 8 workers : roughly 5-16 minutes


#### Streaming to cloud storage, one species at a time

Local disk cannot hold 1.9 GB here, so the full image set is never materialised locally. Each species is downloaded, archived, uploaded, and deleted before the next begins — peak local usage is one species, roughly 66 MB.

Sharding by **species** rather than by fixed-size batch is deliberate. Drive and object stores are slow at per-file operations, and a training run that reads 15,088 individual files from a mounted remote spends most of its time on metadata rather than computation. Thirty-three archives upload in minutes and extract in seconds — and because the shard boundary is the class boundary, each archive unpacks straight into the `ImageFolder` layout, and any single species can be fetched without the rest.

**The manifest, not the filesystem, is the ledger.** Local files are deleted after upload, so "does the file exist?" no longer means "have we fetched it?". `download()` filters against completed manifest rows *before* touching the disk, so a deleted-but-recorded image is never re-fetched.

##### Prerequisite: install rclone (terminal, not a notebook cell)

> ⚠️ **`pip install rclone` does not install rclone.** rclone is a Go binary, not a Python package. The PyPI project of that name is an unrelated thin wrapper that still requires the real binary — installing it is why `shutil.which("rclone")` kept returning `None`. Remove it to avoid confusion:
>
> ```bash
> pip uninstall -y rclone
> ```

The real installation, and the one-time remote setup:

```bash
brew install rclone
rclone config        # new remote named "gdrive", storage "drive", scope 3
rclone lsd gdrive:   # verify it authenticated
rclone mkdir gdrive:insects-dataset/images
```

Scope 3 (`drive.file`) lets rclone see only files it created, rather than your entire Drive — the narrowest permission that does the job, the same instinct behind this project's licence filtering.

**Dependencies are checked, not asserted, below.** A hard `assert` at definition time makes the whole notebook un-runnable on a machine without rclone, including for anyone reading it who only wants the analysis. The check reports the problem; the download cell refuses to start without it.

In [35]:
import shutil
import subprocess

REMOTE = "gdrive:insects-dataset/images"

# Soft check, not an assert: the notebook must stay runnable without rclone.
RCLONE = shutil.which("rclone")
if RCLONE is None:
    print("rclone not on PATH — see the cell above (brew install rclone).")
else:
    print(f"rclone: {RCLONE}")


def upload_species(species: str, remote: str = REMOTE) -> str:
    """Archive one species folder, copy it to the remote, then remove both locally."""
    slug = species.replace(" ", "_")
    folder = config.IMAGES / slug
    if not folder.exists():
        return "skipped: no local folder"

    archive = shutil.make_archive(
        base_name=str(config.IMAGES / slug),
        format="zip",
        root_dir=config.IMAGES,   # archive is built from here...
        base_dir=slug,            # ...and stores paths as slug/<file>.jpg
    )

    # List form, never shell=True: a species name can never become shell syntax.
    # check=True matters here — without it a failed upload would still fall
    # through to the delete below and destroy the only copy.
    subprocess.run(
        [RCLONE, "copy", archive, remote, "--transfers", "4", "--retries", "3"],
        check=True,
        capture_output=True,
    )

    size_mb = Path(archive).stat().st_size / 1024**2   # read size before deleting
    shutil.rmtree(folder)
    Path(archive).unlink()
    return f"uploaded {slug}.zip ({size_mb:.0f} MB)"

rclone: /opt/homebrew/bin/rclone


In [36]:
# Guarded on purpose: this fetches ~1.9 GB and uploads it. A "Run All" must not
# trigger it by accident. Set to True deliberately, once rclone is configured
# and there is disk headroom.
RUN_DOWNLOAD = False

if not RUN_DOWNLOAD:
    print("RUN_DOWNLOAD is False — skipping the fetch.")
    print("Prerequisites: brew install rclone && rclone config, then set it True.")
elif RCLONE is None:
    print("rclone is not installed — cannot upload. Nothing was downloaded.")
else:
    from insects.download import download

    logging.getLogger("insects.download").setLevel(logging.WARNING)

    for species in tqdm(sorted(butterflies_all["label_species"].unique()),
                        desc="species"):
        rows = butterflies_all[butterflies_all["label_species"] == species]
        download(rows)                      # writes/updates Data/image_manifest.csv
        print(f"{species:26} {len(rows):>4} imgs | {upload_species(species)}")

    print("\ndone — manifest at", config.MANIFEST_CSV.relative_to(config.ROOT))

RUN_DOWNLOAD is False — skipping the fetch.
Prerequisites: brew install rclone && rclone config, then set it True.


### 3.4 Verifying the manifest

The manifest is the dataset's ledger, so it is checked rather than trusted. Four questions, each with a wrong answer that would otherwise go unnoticed:

- **How many images actually arrived?** `status` counts `ok` / `cached` / `failed`.
- **Does coverage match expectation?** The number of distinct `photo_id` values must equal the number of rows we set out to fetch.
- **Are any two files byte-identical?** Deduplication so far has been on `photo_id`, but the same photograph can be uploaded twice under different ids. Identical `sha256` values reveal that — and duplicates straddling a train/test boundary are leakage.
- **What failed, and why?** Failures were captured as data rather than raised, so they can be read here. Re-running 3.3 retries exactly those rows.

This cell is written to run **before** the download too: it reports that no manifest exists and moves on, so a full top-to-bottom execution never dies on a missing file.

In [37]:
if not config.MANIFEST_CSV.exists():
    print("No manifest yet — run the cell above with RUN_DOWNLOAD = True.")
    print("Section 4 does not depend on this and can be run now.")
else:
    manifest = pd.read_csv(config.MANIFEST_CSV)

    print(manifest["status"].value_counts().to_string())
    print(f"\ntotal size : {manifest['bytes'].sum() / 1024**3:.2f} GB")
    print(f"coverage   : {manifest['photo_id'].nunique():,} of "
          f"{len(butterflies_all):,} expected images")

    # Byte-identical files under different photo ids are a leakage risk that
    # deduplicating on photo_id cannot catch.
    dupes = manifest.loc[manifest["sha256"].notna(), "sha256"].duplicated().sum()
    print(f"duplicate sha256: {dupes:,}")

    failed = manifest[manifest["status"] == "failed"]
    if len(failed):
        print(f"\n{len(failed)} failures — re-running 3.3 retries only these:")
        print(failed["error"].value_counts().head().to_string())

No manifest yet — run the cell above with RUN_DOWNLOAD = True.
Section 4 does not depend on this and can be run now.


_______________
## 4. The Butterfly Image Dataset

### 4.1 What this dataset is

`butterfly_images.csv` is the **image-level deliverable**: one row per image file, carrying everything needed to load that file, know what it depicts, and honour its licence.

It is deliberately *not* the same object as `butterflies_all.csv`. That table is a record of **observations and their photos as published by iNaturalist**. This one is a record of **files in our image store**. Keeping them separate means the harvest can be re-run without disturbing the image set, and the image set can be re-materialised without re-querying the API.

Three columns describe where an image lives, and they are derived rather than typed by hand:

| Column | Example | Role |
|---|---|---|
| `class_dir` | `Vanessa_cardui` | the class label, as a directory name |
| `local_path` | `images/Vanessa_cardui/709822987.jpg` | path relative to the repo root |
| `archive` | `Vanessa_cardui.zip` | the shard the file was uploaded in |

`local_path` is computed with `insects.download.target_path` — the **same function the downloader uses** to decide where to write. Re-deriving the path with a second expression here would work today and silently break the day the layout changes. When two parts of a system must agree on a value, they should call the same function, not compute the same thing twice.

This cell works **whether or not the images have been downloaded**. The paths are deterministic, so the dataset can be built now and the integrity columns (`bytes`, `sha256`, `status`) fill in from the manifest once section 3.3 has run.

In [38]:
from insects.download import target_path

images = butterflies_all.copy()

# --- where the bytes live ------------------------------------------------
images["class_dir"] = images["label_species"].str.replace(" ", "_", regex=False)
images["file_name"] = images["photo_id"].astype(str) + ".jpg"
images["image_id"] = images["class_dir"] + "/" + images["file_name"]
images["archive"] = images["class_dir"] + ".zip"
images["local_path"] = [
    str(target_path(species, pid, config.IMAGES).relative_to(config.ROOT))
    for species, pid in zip(images["label_species"], images["photo_id"])
]

# --- label granularity ---------------------------------------------------
# taxon_name is already the finest identification available (a trinomial where
# the observer identified a subspecies, the binomial otherwise), so it is used
# as-is rather than duplicated under a new name.
images["rank_identified"] = np.where(
    images["subspecies"].notna(), "subspecies", "species")

# --- rights --------------------------------------------------------------
images["commercial_ok"] = images["license_code"].isin(config.COMMERCIAL_SAFE)

# --- integrity, from the download manifest when it exists ----------------
MANIFEST_FIELDS = ["bytes", "sha256", "status"]
if config.MANIFEST_CSV.exists():
    manifest_cols = pd.read_csv(config.MANIFEST_CSV)[["photo_id"] + MANIFEST_FIELDS]
    images = images.merge(manifest_cols, on="photo_id", how="left",
                          validate="one_to_one")
    print(f"joined manifest: {images['status'].notna().sum():,} of "
          f"{len(images):,} images materialised")
else:
    for col in MANIFEST_FIELDS:
        images[col] = pd.NA
    print("no manifest yet — bytes/sha256/status stay empty until 3.3 has run")

assert images["image_id"].is_unique, "image_id is not unique"

print(f"{len(images):,} rows prepared")
images[["image_id", "label_species", "subspecies", "taxon_name", "local_path"]].head()

no manifest yet — bytes/sha256/status stay empty until 3.3 has run
15,088 rows prepared


,image_id,label_species,subspecies,taxon_name,local_path
0,Pararge_aegeria/707990559.jpg,Pararge aegeria,None,Pararge aegeria,images/Pararge_aegeria/707990559.jpg
1,Lycaena_phlaeas/707988568.jpg,Lycaena phlaeas,None,Lycaena phlaeas,images/Lycaena_phlaeas/707988568.jpg
2,Lycaena_phlaeas/707988572.jpg,Lycaena phlaeas,None,Lycaena phlaeas,images/Lycaena_phlaeas/707988572.jpg
3,Pieris_brassicae/705848273.jpg,Pieris brassicae,None,Pieris brassicae,images/Pieris_brassicae/705848273.jpg
4,Papilio_demoleus/705396961.jpg,Papilio demoleus,None,Papilio demoleus,images/Papilio_demoleus/705396961.jpg


### 4.2 How subspecies are represented

The brief asked for a dataset that supports the subspecies name. The data determines *how* that support must be built.

Subspecies are present on **125 of 15,088 images — 0.8%**, covering 14 distinct subspecies across 14 of the 33 species. That number rules out one design and argues for another.

**Rejected: a subspecies folder hierarchy.** `images/<species>/<subspecies>/` would leave 99.2% of images in an `unknown` bucket, split already-small classes into unusable fragments, and break the `ImageFolder` convention that makes the layout worth having. A directory tree is the wrong structure for a 99%-empty attribute.

**Adopted: subspecies as metadata**, with four columns serving four different purposes:

| Column | Value | Purpose |
|---|---|---|
| `label_species` | `Colias crocea` | the **classification target** — complete for every row |
| `subspecies` | `helice` | the epithet alone, `NaN` where unidentified |
| `taxon_name` | `Colias crocea helice` | the **finest identification available**, falling back to the binomial |
| `rank_identified` | `subspecies` / `species` | how precise this row's identification is |

`taxon_name` is the column to use when the subspecies matters — it is never null, and it degrades gracefully to the species name. Note that no new column was invented for this: `taxon_name` already holds exactly that value as published by iNaturalist. **Duplicating an existing column under a friendlier name creates two things that must agree and eventually will not.**

**Why subspecies must not become the training target.** These are geographic races, so subspecies identity is strongly confounded with location — and location is confounded with photographer, camera and habitat. A model trained on it would likely learn the background rather than the butterfly. The honest use of this column is *analysis*: checking whether within-class visual variation tracks subspecies, and whether errors concentrate in non-Cypriot races.

**Store the finest label you have; train on the coarsest label that is complete.**

In [39]:
ssp = images[images["subspecies"].notna()]

print(f"images with a subspecies : {len(ssp):,} / {len(images):,} "
      f"({len(ssp) / len(images):.1%})")
print(f"distinct subspecies      : {ssp['taxon_name'].nunique()}")
print(f"species affected         : {ssp['label_species'].nunique()} / "
      f"{images['label_species'].nunique()}")

(ssp.groupby(["label_species", "taxon_name"])
    .agg(images=("image_id", "size"),
         cyprus=("source", lambda s: int((s == "cyprus_inat").sum())))
    .sort_values("images", ascending=False))

images with a subspecies : 125 / 15,088 (0.8%)
distinct subspecies      : 14
species affected         : 14 / 33


,,images,cyprus
label_species,taxon_name,,
Gegenes pumilio,Gegenes pumilio gambica,79,0
Freyeria trochylus,Freyeria trochylus trochylus,10,0
Celastrina argiolus,Celastrina argiolus ladonides,9,0
Lasiommata maera,Lasiommata maera maera,7,0
Tarucus balkanica,Tarucus balkanica nigra,6,0
Papilio demoleus,Papilio demoleus malayanus,3,0
Colias crocea,Colias crocea helice,2,0
Pieris brassicae,Pieris brassicae azorensis,2,0
Pieris rapae,Pieris rapae crucivora,2,0


### 4.3 Writing the dataset and its datasheet

The column order below is not cosmetic. A schema that reads **identity → labels (coarse to fine) → storage → rights → provenance** tells a new reader what the table *is* before they have read any documentation. Datasets are read by people who did not build them, including you in three months.

Two columns carry the rights, and they are not optional. Every iNaturalist photo is used under a Creative Commons licence that **requires attribution**, so `license_code` and `attribution` travel with each row rather than sitting in a separate file that can be lost in a copy. `commercial_ok` pre-computes the distinction that actually restricts reuse.

Alongside the CSV we generate a **datasheet** — a short, structured document stating what the dataset contains, how it was collected, how it is licensed and how it should be used. The format follows Gebru et al., *Datasheets for Datasets* (Communications of the ACM 64(12), 2021, [doi:10.1145/3458723](https://doi.org/10.1145/3458723)), now the standard accompaniment to a released dataset.

It is generated **from the data, by code**, not written by hand. A hand-written datasheet is correct on the day it is written and wrong after the next pipeline change; a generated one cannot drift from the dataset it describes.

In [40]:
IMAGES_CSV = BUTTERFLIES_DIR / "butterfly_images.csv"

IMAGE_COLS = [
    # identity
    "image_id", "photo_id", "observation_id",
    # labels, coarse -> fine
    "family", "genus", "label_species", "subspecies",
    "taxon_name", "taxon_id", "taxon_rank", "rank_identified",
    # where the bytes live
    "class_dir", "file_name", "local_path", "archive",
    # remote source, kept so any image can be re-fetched
    "photo_url_medium",
    # rights
    "license_code", "commercial_ok", "attribution", "photographer",
    # provenance and integrity
    "source", "location", "district", "bytes", "sha256", "status",
]

images = images[IMAGE_COLS]
images.to_csv(IMAGES_CSV, index=False)

print(f"wrote {len(images):,} rows x {len(images.columns)} cols "
      f"-> {IMAGES_CSV.relative_to(config.ROOT)}")

per_class = (images.groupby(["family", "label_species"])
             .agg(images=("image_id", "size"),
                  observations=("observation_id", "nunique"),
                  cyprus=("source", lambda s: int((s == "cyprus_inat").sum())),
                  subspecies=("subspecies", "nunique"))
             .sort_values("images", ascending=False))

print(f"\n{len(per_class)} classes")
per_class

wrote 15,088 rows x 26 cols -> Data/ButterfliesOnly/butterfly_images.csv

33 classes


images  observations  cyprus  subspecies
family       label_species                                                   
Hesperiidae  Carcharodus alceae         500           297      76           0
Nymphalidae  Hipparchia syriaca         500           335      52           0
Pieridae     Pieris rapae               500           325     107           1
             Pieris brassicae           500           315     169           1
             Gonepteryx cleopatra       500           303      30           0
             Euchloe ausonia            500           305      21           0
             Colias crocea              500           320     141           1
             Anthocharis cardamines     500           342      36           1
Papilionidae Zerynthia cerisy           500           325      55           0
             Papilio machaon            500           285     290           1
             Papilio demoleus           500           271      77           1
Nymphalidae  Vanessa cardui             500           309     362           0
             Vanessa atalanta           500           312      81           1
             Pararge aegeria            500           340      62           1
             Lasiommata maera           500           314      27           1
Pieridae     Pontia daplidice           500           338     115           0
Hesperiidae  Thymelicus acteon          500           290      66           0
Lycaenidae   Zizeeria karsandra         500           298      81           0
Hesperiidae  Pelopidas thrax            500           282      46           0
Lycaenidae   Tarucus balkanica          500           328      45           1
             Polyommatus icarus         500           332      86           0
             Lycaena thersamon          500           255      42           0
             Lycaena phlaeas            500           316     101           1
Nymphalidae  Chazara briseis            500           346      43           0
Lycaenidae   Leptotes pirithous         500           307      87           0
             Lampides boeticus          500           338      42           0
             Freyeria trochylus         500           306      22           1
             Celastrina argiolus        500           303      58           1
Hesperiidae  Gegenes pumilio            475           316      39           1
Lycaenidae   Glaucopsyche paphos        212           127     189           0
             Luthrodes galba            153            87      33           0
Nymphalidae  Hipparchia cypriensis      132            86     122           0
             Maniola cypricola          116            81      97           0

In [41]:
from datetime import date

CARD_PATH = config.REPORTS / "butterfly_images_datasheet.md"

vc = images["label_species"].value_counts()
prov = " / ".join(f"`{k}` {v:,}" for k, v in images["source"].value_counts().items())
lic_rows = "\n".join(
    f"| `{k}` | {v:,} | {v / len(images):.1%} |"
    for k, v in images["license_code"].value_counts().items()
)

card = f"""# Datasheet — `butterfly_images`

*Generated {date.today().isoformat()} by `Analysis/DataEngineering.ipynb`. Do not edit by hand.*

## Motivation
Fine-grained image classification of the butterflies (superfamily Papilionoidea)
recorded in Cyprus. Built for an academic project supervised by
Dr. Demetris Trihinas. Not intended for commercial use.

## Composition
- **{len(images):,}** images across **{images['label_species'].nunique()}** species,
  **{images['family'].nunique()}** families, **{images['observation_id'].nunique():,}** observations.
- Class balance: max **{vc.max()}**, median **{int(vc.median())}**, min **{vc.min()}**
  — a ratio of **{vc.max() / vc.min():.1f}:1**.
- Subspecies recorded for **{int(images['subspecies'].notna().sum())}** images
  ({images['subspecies'].nunique()} distinct subspecies across
  {images.loc[images['subspecies'].notna(), 'label_species'].nunique()} species).
- Provenance: {prov}.
- One row per image; files are sharded into {images['archive'].nunique()} per-species zip archives.

## Collection process
Observations were retrieved from the iNaturalist API (`/v1/observations`),
restricted to `quality_grade=research` and to openly licensed photos.
Class membership was determined **taxonomically**, by testing whether iNaturalist
taxon **47224** (Papilionoidea) appears in each record's `ancestor_ids` — never by
matching species names. The circumscription follows Heikkilä et al. (2012,
doi:10.1098/rspb.2011.1430) and Kawahara et al. (2023, doi:10.1038/s41559-023-02041-9),
and therefore includes skippers (Hesperiidae).

Cyprus records came through the audited pipeline (`insects_clean.csv`). Additional
images were harvested worldwide **by `taxon_id`**, capped at {HARVEST_CAP} images per
species to limit class imbalance.

## Licensing
All images are openly licensed; every row carries `license_code` and `attribution`,
which must be preserved on redistribution.

| Licence | Images | Share |
|---|---:|---:|
{lic_rows}

**Only {int(images['commercial_ok'].sum()):,} images ({images['commercial_ok'].mean():.1%}) permit commercial use.**
Filter on `commercial_ok` before any non-academic reuse.

## Recommended use
- Target variable: `label_species` ({images['label_species'].nunique()} classes).
- **Split grouped by `observation_id`** — multiple photos of one individual exist by
  design and must not straddle train/test (audit finding S5).
- Evaluate on `source == "cyprus_inat"` to measure performance on the actual task.

## Limitations
See section 4.4 of the generating notebook: geographic domain shift (81% non-Cyprus),
residual imbalance driven by Cyprus endemics, recency bias in the harvest,
NonCommercial licensing, uneven subspecies coverage, and the absence of
image-content validation.

*Structure follows Gebru et al., "Datasheets for Datasets" (Communications of the ACM, 2021).*
"""

CARD_PATH.parent.mkdir(parents=True, exist_ok=True)
CARD_PATH.write_text(card, encoding="utf-8")
print(f"wrote datasheet -> {CARD_PATH.relative_to(config.ROOT)} ({len(card):,} chars)")

wrote datasheet -> Reports/butterfly_images_datasheet.md (2,583 chars)


### 4.4 Known limitations

Recorded here because a dataset whose weaknesses are documented can be used carefully, while one whose weaknesses are undocumented will be used wrongly.

1. **Geographic domain shift.** 12,188 of 15,088 images (81%) come from outside Cyprus. Subspecies such as *Papilio machaon hudsonianus* (North America) and *Pieris brassicae azorensis* (Azores) differ visibly from Cypriot populations, as do habitat, vegetation and light. The `source` column exists so a model may be trained on everything but **evaluated on `cyprus_inat` alone**; reporting accuracy over the mixed set would overstate performance on the actual task.

2. **Residual class imbalance (4.3:1).** The cap fixes the head of the distribution, not the tail: Cyprus endemics (*Glaucopsyche paphos*, *Hipparchia cypriensis*, *Maniola cypricola*) have almost no photographs anywhere in the world, so no amount of harvesting reaches 500. Handle this with class weights or resampling at training time — never by duplicating or generating images.

3. **Recency bias.** The harvest pages by descending observation id, so the added images are the *most recent* qualifying ones. This makes the pull exactly reproducible but not a random sample over time.

4. **Licence restriction.** 85% of the images are NonCommercial (see 4.3). The dataset is usable for this academic project; it is not redistributable commercially.

5. **Uneven label granularity.** Only 0.8% of images carry a subspecies, and identification effort varies by observer and region. `label_species` is the only rank with consistent coverage and therefore the only defensible target.

6. **No image-content validation yet.** Labels are community-verified, but nothing has checked that each file decodes, is not a duplicate under a different `photo_id`, or shows the insect rather than a habitat shot. The `sha256` column supports the duplicate check; that belongs in the next notebook.

### Next steps

The engineering stage ends here. What follows is modelling work:

    butterfly_images.csv
      → verify decoding + drop sha256 duplicates
      → grouped split by observation_id (audit S5)   -> Data/splits.csv
      → baseline transfer-learning classifier
      → evaluate on source == "cyprus_inat" only

**The split must be grouped by `observation_id`.** Several photos of the same individual butterfly exist in this table by design; if they land on both sides of the split, the test score measures memorisation rather than generalisation. This is audit finding S5, and it is the single easiest way to invalidate every number in the final report.